<div style="background: linear-gradient(135deg, #0f172a, #1d4ed8); padding: 24px; border-radius: 16px; color: white; margin-bottom: 20px;">
<h1 style="margin: 0;">Feature Engineering — Base 3</h1>
<p style="margin-top: 10px;">Esta etapa constrói a matriz exploratória de 671 colunas. Random Forest e PLS não usam este arquivo. A matriz compartilhada por esses dois modelos está em <code>06_matriz_de_modelagem.ipynb</code> e em <code>features_modelagem_base3.csv</code>.</p>
</div>

**O que este notebook grava em `features_base3.csv`:**
- alvo: `PM2.5(t+1)` da estação Aotizhongxin;
- lags do alvo a partir de `t-1` (`shift` começando em 1);
- médias e desvios móveis calculados depois de `shift(1)`, então a observação da origem não entra na janela;
- variáveis numéricas locais e das estações Changping, Guanyuan e Wanliu;
- calendário do instante previsto.

**O que os modelos usam:** 17 preditores, com as medições da origem `t` incluídas e sem a meteorologia de Guanyuan, que repete a estação local. Nenhum valor de `t+1` entra como preditor.

<div style="background:#eff6ff; border-left: 5px solid #2563eb; padding: 14px; border-radius: 8px; margin: 16px 0;">
<h2 style="margin-top:0; color:#1e3a8a;">1. Contrato de informação</h2>
<p style="margin-bottom:0;">Neste arquivo exploratório, o alvo é <code>PM2.5(t+1)</code> e os lags do próprio PM2.5 começam em <code>t-1</code>. A matriz de modelagem, no notebook seguinte, inclui a medição da origem <code>t</code>. O calendário é o de <code>t+1</code>, conhecido antes da previsão. Nenhum valor realizado de <code>t+1</code> entra como preditor.</p>
</div>

In [9]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

RANDOM_STATE = 42
TARGET = "target_pm25"
HORIZON = 1
LAGS = [1, 2, 3, 6, 12, 24, 168]
WINDOWS = [3, 6, 24, 168]
SOURCE = "https://archive.ics.uci.edu/dataset/501/beijing+multi+site+air+quality+data"


def find_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "bases" / "grupo3").exists():
            return candidate
    raise FileNotFoundError("Raiz do repositório não encontrada.")


ROOT = find_root(Path.cwd().resolve())
OUTPUT_DIR = ROOT / "analyses" / "grupo3" / "01_dados" / "outputs"
INPUT_PATH = OUTPUT_DIR / "prepared_base3.csv"
FEATURE_PATH = OUTPUT_DIR / "features_base3.csv"


<div style="background:#eff6ff; border-left: 5px solid #2563eb; padding: 14px; border-radius: 8px; margin: 16px 0;">
<h2 style="margin-top:0; color:#1e3a8a;">2. Leitura da base preparada e seleção das fontes</h2>
<p style="margin-bottom:0;">São usadas as colunas numéricas preparadas. Indicadores de ausência, calendário bruto, timestamp e o alvo contemporâneo são excluídos da matriz de preditores; o alvo entra somente por lags.</p>
</div>

In [10]:
prepared = pd.read_csv(INPUT_PATH, parse_dates=["timestamp"])
prepared = prepared.sort_values("timestamp", kind="stable").reset_index(drop=True)
if not prepared["timestamp"].is_unique or not prepared["timestamp"].is_monotonic_increasing:
    raise ValueError("A base preparada não está temporalmente ordenada e única.")

calendar_columns = {"timestamp", "year", "month", "day", "hour", "dayofweek", "is_weekend"}
raw_missing_columns = {column for column in prepared.columns if column.endswith("_missing_raw")}
numeric_measurements = [
    column for column in prepared.columns
    if column not in calendar_columns | raw_missing_columns | {TARGET}
    and pd.api.types.is_numeric_dtype(prepared[column])
]
if not numeric_measurements:
    raise ValueError("Nenhuma medição numérica elegível foi encontrada.")

print(f"Observações preparadas: {len(prepared):,}")
print(f"Medições numéricas elegíveis: {len(numeric_measurements)}")
print(numeric_measurements)


Observações preparadas: 35,064
Medições numéricas elegíveis: 43
['local_PM10', 'local_SO2', 'local_NO2', 'local_CO', 'local_O3', 'local_TEMP', 'local_PRES', 'local_DEWP', 'local_RAIN', 'local_WSPM', 'Changping_PM2.5', 'Changping_PM10', 'Changping_SO2', 'Changping_NO2', 'Changping_CO', 'Changping_O3', 'Changping_TEMP', 'Changping_PRES', 'Changping_DEWP', 'Changping_RAIN', 'Changping_WSPM', 'Guanyuan_PM2.5', 'Guanyuan_PM10', 'Guanyuan_SO2', 'Guanyuan_NO2', 'Guanyuan_CO', 'Guanyuan_O3', 'Guanyuan_TEMP', 'Guanyuan_PRES', 'Guanyuan_DEWP', 'Guanyuan_RAIN', 'Guanyuan_WSPM', 'Wanliu_PM2.5', 'Wanliu_PM10', 'Wanliu_SO2', 'Wanliu_NO2', 'Wanliu_CO', 'Wanliu_O3', 'Wanliu_TEMP', 'Wanliu_PRES', 'Wanliu_DEWP', 'Wanliu_RAIN', 'Wanliu_WSPM']


<div style="background:#eff6ff; border-left: 5px solid #2563eb; padding: 14px; border-radius: 8px; margin: 16px 0;">
<h2 style="margin-top:0; color:#1e3a8a;">3. Construção de lags e estatísticas causais</h2>
<p style="margin-bottom:0;">Cada lag usa <code>shift(lag)</code>. Cada média ou desvio usa primeiro <code>shift(1)</code> e somente depois a janela móvel; portanto, a observação atual não participa da estatística usada na própria origem.</p>
</div>

In [11]:
# O alvo futuro nunca é imputado: sem valor observado, não há rótulo para treinar.
feature_frame = pd.DataFrame(index=prepared.index)
feature_frame["timestamp"] = prepared["timestamp"]
feature_frame["target_pm25_t_plus_1"] = prepared[TARGET].shift(-HORIZON)
feature_frame["target_timestamp"] = prepared["timestamp"].shift(-HORIZON)

# Histórico do alvo: preenchimento causal limitado para não carregar valor antigo indefinidamente.
TARGET_HISTORY_FFILL_LIMIT = 24
raw_target_missing = prepared[TARGET].isna()
target_history = prepared[TARGET].ffill(limit=TARGET_HISTORY_FFILL_LIMIT)
target_missing_age = raw_target_missing.astype("int8").groupby((~raw_target_missing).cumsum()).cumsum()
feature_frame["target_history_imputed"] = raw_target_missing.astype("int8")
feature_frame["target_history_imputed_age"] = target_missing_age.clip(upper=TARGET_HISTORY_FFILL_LIMIT).astype("int16")

for lag in LAGS:
    feature_frame[f"target_pm25_lag_{lag}h"] = target_history.shift(lag)

# Medições numéricas locais e externas, todas defasadas.
for column in numeric_measurements:
    safe_name = column.replace(".", "_")
    for lag in LAGS:
        feature_frame[f"{safe_name}_lag_{lag}h"] = prepared[column].shift(lag)

# Estatísticas de janela do PM2.5 e das medições numéricas, encerradas em t-1.
for column in [TARGET] + numeric_measurements:
    safe_name = column.replace(".", "_")
    historical = target_history.shift(1) if column == TARGET else prepared[column].shift(1)
    for window in WINDOWS:
        feature_frame[f"{safe_name}_rolling_mean_{window}h"] = historical.rolling(window=window, min_periods=window).mean()
        feature_frame[f"{safe_name}_rolling_std_{window}h"] = historical.rolling(window=window, min_periods=window).std(ddof=0)

# Calendário do instante previsto: informação determinística conhecida antes de t+1.
target_time = prepared["timestamp"].shift(-HORIZON)
feature_frame["target_hour"] = target_time.dt.hour
feature_frame["target_dayofweek"] = target_time.dt.dayofweek
feature_frame["target_is_weekend"] = target_time.dt.dayofweek.isin([5, 6]).astype("int8")
feature_frame["target_hour_sin"] = np.sin(2 * np.pi * target_time.dt.hour / 24)
feature_frame["target_hour_cos"] = np.cos(2 * np.pi * target_time.dt.hour / 24)
feature_frame["target_dayofweek_sin"] = np.sin(2 * np.pi * target_time.dt.dayofweek / 7)
feature_frame["target_dayofweek_cos"] = np.cos(2 * np.pi * target_time.dt.dayofweek / 7)
feature_frame["target_month_sin"] = np.sin(2 * np.pi * target_time.dt.month / 12)
feature_frame["target_month_cos"] = np.cos(2 * np.pi * target_time.dt.month / 12)

# Remove apenas linhas sem rótulo ou com histórico ainda indisponível; sem dropna global.
feature_frame = feature_frame.loc[feature_frame["target_pm25_t_plus_1"].notna()].copy()
feature_frame = feature_frame.dropna(subset=[c for c in feature_frame.columns if c != "target_history_imputed_age"]).reset_index(drop=True)
feature_frame.to_csv(FEATURE_PATH, index=False)
print(f"Matriz final: {feature_frame.shape}")
display(feature_frame.head(3))

/var/folders/wq/3jxnkjv16xd1lzf739cr0g3m0000gr/T/ipykernel_69237/2170217629.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  feature_frame[f"{safe_name}_lag_{lag}h"] = prepared[column].shift(lag)
/var/folders/wq/3jxnkjv16xd1lzf739cr0g3m0000gr/T/ipykernel_69237/2170217629.py:22: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  feature_frame[f"{safe_name}_lag_{lag}h"] = prepared[column].shift(lag)
/var/folders/wq/3jxnkjv16xd1lzf739cr0g3m0000gr/T/ipykernel_69237/2170217629.py:22: PerformanceWarning: DataFrame is highly fragmented.

Matriz final: (33306, 674)


,timestamp,target_pm25_t_plus_1,target_timestamp,target_history_imputed,target_history_imputed_age,target_pm25_lag_1h,target_pm25_lag_2h,target_pm25_lag_3h,target_pm25_lag_6h,target_pm25_lag_12h,...,Wanliu_WSPM_rolling_std_168h,target_hour,target_dayofweek,target_is_weekend,target_hour_sin,target_hour_cos,target_dayofweek_sin,target_dayofweek_cos,target_month_sin,target_month_cos
0,2013-03-08 00:00:00,339.0,2013-03-08 01:00:00,0,0,376.0,363.0,282.0,201.0,255.0,...,1.108461,1.0,4.0,0,0.258819,0.965926,-0.433884,-0.900969,1.0,6.123234e-17
1,2013-03-08 01:00:00,339.0,2013-03-08 02:00:00,0,0,344.0,376.0,363.0,211.0,258.0,...,1.099905,2.0,4.0,0,0.500000,0.866025,-0.433884,-0.900969,1.0,6.123234e-17
2,2013-03-08 02:00:00,324.0,2013-03-08 03:00:00,0,0,339.0,344.0,376.0,227.0,277.0,...,1.079456,3.0,4.0,0,0.707107,0.707107,-0.433884,-0.900969,1.0,6.123234e-17


<div style="background:#eff6ff; border-left: 5px solid #2563eb; padding: 14px; border-radius: 8px; margin: 16px 0;">
<h2 style="margin-top:0; color:#1e3a8a;">4. Dicionário, auditoria e manifesto</h2>
<p style="margin-bottom:0;">O dicionário identifica a origem, fórmula, janela e disponibilidade de cada coluna. A auditoria verifica que toda feature termina no passado em relação ao timestamp-alvo.</p>
</div>

In [12]:
dictionary_rows = [
    {"feature": "timestamp", "tipo": "origem da previsão", "origem": "timestamp", "formula": "timestamp(t)", "disponibilidade": "instante da origem"},
    {"feature": "target_pm25_t_plus_1", "tipo": "alvo", "origem": "Aotizhongxin.PM2.5", "formula": "PM2.5(t+1)", "disponibilidade": "rótulo fora da amostra; não usado como preditor"},
    {"feature": "target_timestamp", "tipo": "índice do alvo", "origem": "timestamp", "formula": "timestamp(t+1)", "disponibilidade": "calendário da previsão"},
]
for lag in LAGS:
    dictionary_rows.append({"feature": f"target_pm25_lag_{lag}h", "tipo": "lag do alvo", "origem": "Aotizhongxin.PM2.5", "formula": f"PM2.5(t-{lag})", "disponibilidade": "disponível em t"})
for column in [TARGET] + numeric_measurements:
    safe_name = column.replace(".", "_")
    feature_type = "lag do alvo" if column == TARGET else "lag exógeno"
    rolling_type = "estatística móvel do alvo" if column == TARGET else "estatística móvel exógena"
    if column != TARGET:
        for lag in LAGS:
            dictionary_rows.append({"feature": f"{safe_name}_lag_{lag}h", "tipo": feature_type, "origem": column, "formula": f"{column}(t-{lag})", "disponibilidade": "disponível em t"})
    for window in WINDOWS:
        dictionary_rows.extend([
            {"feature": f"{safe_name}_rolling_mean_{window}h", "tipo": rolling_type, "origem": column, "formula": f"mean({column}(t-{window}:t-1))", "disponibilidade": "disponível em t"},
            {"feature": f"{safe_name}_rolling_std_{window}h", "tipo": rolling_type, "origem": column, "formula": f"std({column}(t-{window}:t-1))", "disponibilidade": "disponível em t"},
        ])
for name in ["target_hour", "target_dayofweek", "target_is_weekend", "target_hour_sin", "target_hour_cos", "target_dayofweek_sin", "target_dayofweek_cos", "target_month_sin", "target_month_cos"]:
    dictionary_rows.append({"feature": name, "tipo": "calendário", "origem": "target_timestamp", "formula": "calendário de t+1", "disponibilidade": "conhecido antecipadamente"})

# Flags de imputação do histórico do alvo
dictionary_rows.append({"feature": "target_history_imputed", "tipo": "indicador de imputação", "origem": "Aotizhongxin.PM2.5", "formula": "1 se PM2.5(t) era ausente no bruto", "disponibilidade": "disponível em t"})
dictionary_rows.append({"feature": "target_history_imputed_age", "tipo": "idade da imputação", "origem": "Aotizhongxin.PM2.5", "formula": "horas consecutivas desde último valor observado", "disponibilidade": "disponível em t"})
dictionary = pd.DataFrame(dictionary_rows)
dictionary.to_csv(OUTPUT_DIR / "feature_dictionary.csv", index=False)

predictor_columns = [column for column in feature_frame.columns if column not in {"timestamp", "target_timestamp", "target_pm25_t_plus_1"}]
assert predictor_columns
assert feature_frame[predictor_columns].isna().sum().sum() == 0
assert feature_frame["target_pm25_t_plus_1"].notna().all()
assert feature_frame["target_timestamp"].gt(feature_frame["timestamp"]).all()
assert len(dictionary) == len(feature_frame.columns)

manifest = {
    "source": SOURCE,
    "input_file": str(INPUT_PATH.relative_to(ROOT)),
    "output_file": str(FEATURE_PATH.relative_to(ROOT)),
    "target": "target_pm25_t_plus_1",
    "origin_timestamp": "timestamp",
    "horizon_hours": HORIZON,
    "rows": int(len(feature_frame)),
    "columns": int(len(feature_frame.columns)),
    "predictor_columns": int(len(predictor_columns)),
    "lags_hours": LAGS,
    "rolling_windows_hours": WINDOWS,
    "external_scope": "all numeric local and external measurements from prepared_base3.csv",
    "categorical_scope": "wind direction excluded from shared numeric matrix",
    "rolling_formula": "shift(1).rolling(window, min_periods=window)",
    "calendar_policy": "calendar features use t+1 and are known in advance",
    "leakage_checks": {
        "no_current_value_in_rolling_window": True,
        "all_measurements_lagged": True,
        "target_not_used_as_contemporaneous_predictor": True,
        "future_target_only_as_label": True,
    },
    "random_state": RANDOM_STATE,
}
with (OUTPUT_DIR / "feature_manifest.json").open("w", encoding="utf-8") as file:
    json.dump(manifest, file, ensure_ascii=False, indent=2)

display(dictionary.head(12))
print(f"Features salvas em: {FEATURE_PATH}")
print(f"Preditores: {len(predictor_columns)} | linhas utilizáveis: {len(feature_frame):,}")


,feature,tipo,origem,formula,disponibilidade
0,timestamp,origem da previsão,timestamp,timestamp(t),instante da origem
1,target_pm25_t_plus_1,alvo,Aotizhongxin.PM2.5,PM2.5(t+1),rótulo fora da amostra; não usado como preditor
2,target_timestamp,índice do alvo,timestamp,timestamp(t+1),calendário da previsão
3,target_pm25_lag_1h,lag do alvo,Aotizhongxin.PM2.5,PM2.5(t-1),disponível em t
4,target_pm25_lag_2h,lag do alvo,Aotizhongxin.PM2.5,PM2.5(t-2),disponível em t
5,target_pm25_lag_3h,lag do alvo,Aotizhongxin.PM2.5,PM2.5(t-3),disponível em t
6,target_pm25_lag_6h,lag do alvo,Aotizhongxin.PM2.5,PM2.5(t-6),disponível em t
7,target_pm25_lag_12h,lag do alvo,Aotizhongxin.PM2.5,PM2.5(t-12),disponível em t
8,target_pm25_lag_24h,lag do alvo,Aotizhongxin.PM2.5,PM2.5(t-24),disponível em t
9,target_pm25_lag_168h,lag do alvo,Aotizhongxin.PM2.5,PM2.5(t-168),disponível em t


Features salvas em: /Users/leticia.nsilva/Desktop/personal/pls-regration-comparation/analyses/grupo3/01_dados/outputs/features_base3.csv
Preditores: 671 | linhas utilizáveis: 33,306


<div style="background:#dbeafe; border-radius: 12px; padding: 16px; color:#1e3a8a;">
<h2 style="margin-top:0;">Validação da Fase 3</h2>
<ul style="margin-bottom:0;">
<li>A matriz é comum aos modelos tabulares Random Forest e PLS Regression.</li>
<li>As primeiras observações são removidas apenas porque não possuem histórico suficiente para o maior lag/janela.</li>
<li>Linhas com alvo ausente são removidas por não permitirem avaliação supervisionada; nenhum alvo ausente foi imputado.</li>
<li>O teste 80/20 e o ajuste de escalonadores pertencem à etapa de modelagem e não são aplicados aqui.</li>
</ul>
</div>